In [2]:
import pyspark as py

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-silver") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df_orders = spark.read.parquet("../data/bronze/orders/")
df_customers = spark.read.parquet("../data/bronze/customers/")
df_items = spark.read.parquet("../data/bronze/order_items/")
df_payments = spark.read.parquet("../data/bronze/order_payments/")
df_reviews = spark.read.parquet("../data/bronze/order_reviews/")
df_products = spark.read.parquet("../data/bronze/products/")
df_sellers = spark.read.parquet("../data/bronze/sellers/")
df_geo = spark.read.parquet("../data/bronze/geolocation/")
df_pcnt = spark.read.parquet("../data/bronze/product_category_name_translation/")

dataframes = {
    "orders": df_orders,
    "customers": df_customers,
    "items": df_items,
    "payments": df_payments,
    "reviews": df_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geo": df_geo,
    "pcnt": df_pcnt
}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 14:25:50 WARN Utils: Your hostname, edilene-ThinkPad-P53, resolves to a loopback address: 127.0.1.1; using 10.26.1.68 instead (on interface wlp82s0)
26/06/17 14:25:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 14:25:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/17 14:25:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [7]:
from pyspark.sql.functions import col, sum as _sum, count, when

def audit_nulls(df, name):
    null_counts = df.select([_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
    null_counts.show()
    print(f"DataFrame: {name}")
    print(f"Total Rows: {df.count()}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Null Counts: {null_counts.collect()[0].asDict()}")

for df_name, df in dataframes.items():
    audit_nulls(df, df_name)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

DataFrame: orders
Total Rows: 99441
Total Columns: 8
Null Counts: {'order_id': 0, 'customer_id': 0, 'order_status': 0, 'order_purchase_timestamp': 0, 'order_ap

In [8]:
def audit_duplicates(df, name):
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"DataFrame: {name}")
    print(f"Total Rows: {df.count()}")
    print(f"Duplicate Rows: {duplicate_count}")

for df_name, df in dataframes.items():
    audit_duplicates(df, df_name)

DataFrame: orders
Total Rows: 99441
Duplicate Rows: 0
DataFrame: customers
Total Rows: 99441
Duplicate Rows: 0
DataFrame: items
Total Rows: 112650
Duplicate Rows: 0
DataFrame: payments
Total Rows: 103886
Duplicate Rows: 0
DataFrame: reviews
Total Rows: 99224
Duplicate Rows: 0
DataFrame: products
Total Rows: 32951
Duplicate Rows: 0
DataFrame: sellers
Total Rows: 3095
Duplicate Rows: 0


DataFrame: geo
Total Rows: 1000163
Duplicate Rows: 261831
DataFrame: pcnt
Total Rows: 71
Duplicate Rows: 0


In [9]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType

print("Avant conversion des types de données :")
df_reviews.printSchema()

df_reviews = df_reviews \
    .withColumn("review_score", col("review_score").cast(IntegerType()))

print("Après conversion des types de données :")
df_reviews.printSchema()

Avant conversion des types de données :
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Après conversion des types de données :
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [10]:
df_orders = df_orders.dropna(subset=["order_id", "customer_id"])
df_items = df_items.dropna(subset=["order_id", "product_id"])
df_reviews = df_reviews.dropna(subset=["order_id"])

In [11]:
from pyspark.sql.functions import lit

# Texte manquant dans les avis → remplacer par chaîne vide
df_reviews = df_reviews.fillna({
    "review_comment_title": "",
    "review_comment_message": ""
})

# Catégorie produit inconnue
df_products = df_products.fillna({
    "product_category_name": "unknown"
})

# Valeurs numériques manquantes
df_items = df_items.fillna({"freight_value": 0.0})

In [12]:
# Sur la clé primaire uniquement
df_orders = df_orders.dropDuplicates(["order_id"])
df_customers = df_customers.dropDuplicates(["customer_id"])
df_products = df_products.dropDuplicates(["product_id"])
df_sellers = df_sellers.dropDuplicates(["seller_id"])

# Géolocalisation : doublons fréquents et attendus → dédoublonner sur le zip
df_geo = df_geo.dropDuplicates(["geolocation_zip_code_prefix"])

In [13]:
from pyspark.sql.functions import col

# Commandes livrées AVANT d'être achetées = incohérent
incoherences = df_orders.filter(
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
)
print(f"Livraisons avant achat : {incoherences.count()}")

# Commandes approuvées avant achat
approved_before_buy = df_orders.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
).count()

print(f"Commandes approuvées avant achat : {approved_before_buy}")

Livraisons avant achat : 0
Commandes approuvées avant achat : 1


In [ ]:
print(f"Articles avec prix <= 0 : {df_items.filter(col('price') <= 0).count()}")

print(f"Paiements avec valeur <= 0 : {df_payments.filter(col('payment_value') <= 0).count()}")

# df_payments.filter(col('payment_value') <= 0).show()

Articles avec prix <= 0 : 0
Paiements avec valeur <= 0 : 9
+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|8bcbe01d44d147f90...|                 4|     voucher|                   1|          0.0|
|fa65dad1b0e818e3c...|                14|     voucher|                   1|          0.0|
|6ccb433e00daae128...|                 4|     voucher|                   1|          0.0|
|4637ca194b6387e2d...|                 1| not_defined|                   1|          0.0|
|00b1cb0320190ca0d...|                 1| not_defined|                   1|          0.0|
|45ed6e85398a87c25...|                 3|     voucher|                   1|          0.0|
|fa65dad1b0e818e3c...|                13|     voucher|                   1|          0.0|
|c8c528189310eaa44...|                 1|

In [15]:
print(f"Avis avec score < 1 ou > 5 : {df_reviews.filter((col('review_score') < 1) | (col('review_score') > 5)).count()}")

Avis avec score < 1 ou > 5 : 0


In [16]:
df_orders.write.mode("overwrite").parquet("../data/silver/orders/")
df_customers.write.mode("overwrite").parquet("../data/silver/customers/")
df_items.write.mode("overwrite").parquet("../data/silver/order_items/")
df_payments.write.mode("overwrite").parquet("../data/silver/payments/")
df_reviews.write.mode("overwrite").parquet("../data/silver/reviews/")
df_products.write.mode("overwrite").parquet("../data/silver/products/")
df_sellers.write.mode("overwrite").parquet("../data/silver/sellers/")
df_geo.write.mode("overwrite").parquet("../data/silver/geolocation/")

26/06/17 14:28:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/17 14:28:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/17 14:28:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/17 14:28:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/17 14:28:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/17 14:28:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
